# Dates and Timestamps

Dates often arrive as text. Convert them to Spark temporal types before filtering, grouping, or calculating with them.

---
## Learning objectives

By the end of this notebook, you will be able to:

- parse date and timestamp strings using explicit formats;
- format dates and derive useful calendar fields;
- calculate differences between dates and add time to timestamps; and
- convert a UTC timestamp to a named local time zone.

---
## Start with string values

This delivery dataset is intentionally created with strings. `order_date_text` is a date, and `event_time_utc_text` represents a UTC timestamp.

In [9]:
%run "./0 - Create spark session.ipynb"

In [10]:
from pyspark.sql import functions as F

deliveries_raw = spark.createDataFrame(
    [
        ("O100", "2024-01-15", "2024-01-15 08:30:00"),
        ("O101", "2024-02-01", "2024-02-01 14:45:00"),
        ("O102", "2024-03-10", "2024-03-10 22:15:00"),
    ],
    ["order_id", "order_date_text", "event_time_utc_text"],
)

deliveries_raw.show()
deliveries_raw.printSchema()

+--------+---------------+-------------------+
|order_id|order_date_text|event_time_utc_text|
+--------+---------------+-------------------+
|    O100|     2024-01-15|2024-01-15 08:30:00|
|    O101|     2024-02-01|2024-02-01 14:45:00|
|    O102|     2024-03-10|2024-03-10 22:15:00|
+--------+---------------+-------------------+

root
 |-- order_id: string (nullable = true)
 |-- order_date_text: string (nullable = true)
 |-- event_time_utc_text: string (nullable = true)



---
## Parse strings into temporal types

Use `to_date` and `to_timestamp` with the format that matches the source. `yyyy` means a four-digit calendar year. An explicit format makes the data contract visible and avoids implicit conversion.

In [13]:
deliveries = deliveries_raw.select(
    "order_id",
    F.to_date("order_date_text", "yyyy-MM-dd").alias("order_date"),
    F.to_timestamp("event_time_utc_text", "yyyy-MM-dd HH:mm:ss").alias("event_timestamp_utc"),
)

deliveries.show()
deliveries.printSchema()

+--------+----------+-------------------+
|order_id|order_date|event_timestamp_utc|
+--------+----------+-------------------+
|    O100|2024-01-15|2024-01-15 08:30:00|
|    O101|2024-02-01|2024-02-01 14:45:00|
|    O102|2024-03-10|2024-03-10 22:15:00|
+--------+----------+-------------------+

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- event_timestamp_utc: timestamp (nullable = true)



---
## Format and inspect dates

Keep values as `date` or `timestamp` while transforming them. Use `date_format` only when a human-readable string is needed for output.

In [14]:
date_features = deliveries.select(
    "order_id",
    "order_date",
    F.date_format("order_date", "MM-dd-yyyy").alias("order_date_label"),
    F.year("order_date").alias("order_year"),
    F.month("order_date").alias("order_month"),
    F.dayofmonth("order_date").alias("day_of_month"),
    F.dayofweek("order_date").alias("day_of_week"),
    F.dayofyear("order_date").alias("day_of_year"),
)
date_features.show()

+--------+----------+----------------+----------+-----------+------------+-----------+-----------+
|order_id|order_date|order_date_label|order_year|order_month|day_of_month|day_of_week|day_of_year|
+--------+----------+----------------+----------+-----------+------------+-----------+-----------+
|    O100|2024-01-15|      01-15-2024|      2024|          1|          15|          2|         15|
|    O101|2024-02-01|      02-01-2024|      2024|          2|           1|          5|         32|
|    O102|2024-03-10|      03-10-2024|      2024|          3|          10|          1|         70|
+--------+----------+----------------+----------+-----------+------------+-----------+-----------+



---
## Calculate date and timestamp differences

`datediff` returns the number of whole calendar days between dates. Spark SQL interval syntax is a convenient way to add time to a timestamp.

In [ ]:
delivery_schedule = deliveries.select(
    "order_id",
    "order_date",
    F.datediff(F.current_date(), F.col("order_date")).alias("days_since_order"),
    "event_timestamp_utc",
    F.expr("event_timestamp_utc + INTERVAL 2 HOURS").alias("estimated_delivery_timestamp_utc"),
)
delivery_schedule.show(truncate=False)

---
## Convert UTC to local time

The source timestamps are UTC. `from_utc_timestamp` represents the same instant in a named time zone. Here, operations needs the local time in Paris. In production, use the time zone that belongs to the business event.

In [ ]:
deliveries.select(
    "order_id",
    "event_timestamp_utc",
    F.from_utc_timestamp("event_timestamp_utc", "Europe/Paris").alias("event_timestamp_paris"),
).show(truncate=False)

---
## Your turn: parse a new delivery

Create `new_delivery` from the row below. Convert `ship_date_text` to a `date` named `ship_date` and `pickup_time_utc_text` to a `timestamp` named `pickup_timestamp_utc`. Use `dd/MM/yyyy` and `dd/MM/yyyy HH:mm`, then preview the result and schema.

In [ ]:
new_delivery_raw = spark.createDataFrame(
    [("O103", "25/03/2024", "25/03/2024 09:15")],
    ["order_id", "ship_date_text", "pickup_time_utc_text"],
)

# Write your solution here.

---
## Your turn: create calendar features

Starting from `new_delivery`, select `order_id`, `ship_date`, the shipment month, and the day of year. Name the new fields `ship_month` and `ship_day_of_year`.

In [ ]:
# Write your solution here.